In [1]:
import os, math, argparse, gc, itertools, random
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoProcessor,               # для Whisper
    AutoModelForSpeechSeq2Seq,
    AutoTokenizer,               # для Qwen
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

# from adapters import FCAdapter, TransformerAdapter
from unified_dataset import UnifiedSpeechDataset      # ваш код

In [2]:
import os, math, gc, random, warnings
from typing import List
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from transformers import (
    AutoProcessor, AutoModelForSpeechSeq2Seq,
    AutoTokenizer,  AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from adapters import FCAdapter
from unified_dataset import UnifiedSpeechDataset
from jiwer import wer
warnings.filterwarnings("ignore")

In [3]:
class FCAdapterTiny(FCAdapter):
    def __init__(self):
        super().__init__(m=8, d_x=1280, fc_layer_dim=256, llm_hidden_size=896)

In [4]:
class Speech2Text(nn.Module):
    def __init__(self):
        super().__init__()
        # Whisper encoder
        self.proc = AutoProcessor.from_pretrained("openai/whisper-large-v3")
        self.whisper = AutoModelForSpeechSeq2Seq.from_pretrained(
            "openai/whisper-large-v3",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        ).model.encoder
        for p in self.whisper.parameters():
            p.requires_grad_(False)

        # adapter
        self.adapter = FCAdapterTiny().to(torch.float16)

        # Qwen decoder
        self.tok = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B",
                                                 use_fast=False)
        self.qwen = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2-0.5B",
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True
        )
        for p in self.qwen.parameters():
            p.requires_grad_(False)
        self.txt_emb = self.qwen.get_input_embeddings()

        # fix constants
        self.max_samples = 30 * 16_000          # 30-сек ограничение Whisper

    # ---------- forward (train) ----------------------------------------------
    def forward(self, audios, prompts, responses):
        dev = next(self.adapter.parameters()).device
        # 1 audio → mel → Whisper
        feats = self.proc.feature_extractor(
            audios, sampling_rate=16_000,
            return_tensors="pt", padding="max_length",
            max_length=self.max_samples, truncation=True
        ).input_features.to(dev).to(torch.float16)
        with torch.no_grad():
            h_w = self.whisper(feats).last_hidden_state   # (B,1500,1280)

        # 2 adapter
        h_a = self.adapter(h_w).to(torch.bfloat16)                          # (B,T_a,896)

        # 3 tokenise prompt / response
        ids_p = self.tok(prompts, return_tensors="pt",
                         padding=True, add_special_tokens=False).input_ids.to(dev)
        ids_r = self.tok(responses, return_tensors="pt",
                         padding=True, add_special_tokens=False).input_ids.to(dev)
        with torch.no_grad():
            e_p = self.txt_emb(ids_p)
            e_r = self.txt_emb(ids_r)

        # 4 concat
        inp = torch.cat([h_a, e_p, e_r[:, :-1]], 1)

        # masks
        att = torch.cat([
            torch.ones(h_a.shape[:2], dtype=torch.long, device=dev),
            (ids_p != self.tok.pad_token_id).long(),
            (ids_r[:, :-1] != self.tok.pad_token_id).long()
        ], 1)

        # labels (shifted)
        labels = torch.cat([
            torch.full((ids_p.size(0), h_a.size(1) + ids_p.size(1)),
                       -100, dtype=torch.long, device=dev),
            ids_r[:, 1:]
        ], 1)

        loss = self.qwen(inputs_embeds=inp,
                         attention_mask=att,
                         labels=labels).loss

        return loss

            # ---------- inference для WER --------------------------------------------
    # ───── в классе Speech2Text ─────────────────────────────────────────
    @torch.no_grad()
    def generate_text(self, audio, prompt_str):
        """
        Инференс: аудио + тот же текстовый prompt, что давали на обучении.
        """
        dev = next(self.adapter.parameters()).device
        feats = self.proc.feature_extractor(
            [audio], sampling_rate=16_000, return_tensors="pt",
            padding="max_length", max_length=self.max_samples,
            truncation=True).input_features.to(dev).to(torch.float16)
    
        h_a = self.adapter(self.whisper(feats).last_hidden_state).to(torch.bfloat16)
    
        # prompt → эмбеддинги
        ids_p = self.tok(prompt_str, return_tensors="pt",
                         add_special_tokens=False).input_ids.to(dev)
        e_p   = self.txt_emb(ids_p)

        # ❶ выбираем корректный стартовый id
        start_id = (
            self.tok.bos_token_id
            if self.tok.bos_token_id is not None else
            (self.tok.cls_token_id
             if self.tok.cls_token_id is not None else
             self.tok.eos_token_id)
        )
        
        # ❷ делаем тензор уже с валидным значением
        bos = torch.tensor([[start_id]], device=dev)
        e_bos = self.txt_emb(bos)

        # BOS для генерации
        # bos   = torch.tensor([[self.tok.bos_token_id]], device=dev)
        # e_bos = self.txt_emb(bos)
    
        inp   = torch.cat([h_a, e_p, e_bos], 1)
        att   = torch.cat([
                  torch.ones(h_a.shape[:2], device=dev, dtype=torch.long),
                  torch.ones(ids_p.shape,   device=dev, dtype=torch.long),
                  torch.ones((1,1),         device=dev, dtype=torch.long)
               ], 1)
    
        out = self.qwen.generate(inputs_embeds=inp,
                                 attention_mask=att,
                                 max_new_tokens=128,
                                 pad_token_id=self.tok.pad_token_id,
                                 eos_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0], skip_special_tokens=True)

In [5]:
class Collator:
    def __call__(self, batch):
        return ([x["speech_input"]  for x in batch],
                [x["text_prompt"]   for x in batch],
                [x["text_response"] for x in batch])

In [6]:
import argparse, sys

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--batch_size", type=int, default=8)
    p.add_argument("--epochs",     type=int, default=3)
    p.add_argument("--lr",         type=float, default=2e-7)
    p.add_argument("--output_dir", default="./checkpoints_tiny")
    p.add_argument("--grad_accum", type=int, default=4)
    p.add_argument("--num_workers",type=int, default=4)
    p.add_argument("--max_train_batches", type=int, default=10,  
                   help="0 = весь датасет")
    p.add_argument("--max_val_batches",   type=int, default=5,   
                   help="0 = весь вал-набор")

    args, _ = p.parse_known_args()
    return args

In [7]:
def main():
    args = parse_args()
    dev  = "cuda" if torch.cuda.is_available() else "cpu"

    # train / val splits (val = последние 5 % примеров)
    full_ds = UnifiedSpeechDataset(lang="ru", split="train",
                                   local_files_only=False, batch_size=1)
    val_size = max(1, int(0.05 * len(full_ds)))
    val_ds   = Subset(full_ds, range(len(full_ds) - val_size, len(full_ds)))
    train_ds = Subset(full_ds, range(len(full_ds) - val_size))

    train_loader = DataLoader(train_ds, batch_size=args.batch_size,
                              shuffle=True, collate_fn=Collator(),
                              num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=1,
                              shuffle=False, collate_fn=Collator())

    model = Speech2Text().to(dev)
    opt = AdamW(model.adapter.parameters(), lr=args.lr)
    steps_per_epoch = math.ceil(len(train_loader) / args.grad_accum)
    sched = get_linear_schedule_with_warmup(
        opt, steps_per_epoch//10, steps_per_epoch*args.epochs
    )

    print("Train batches:", len(train_loader),
          "| Val examples:", len(val_loader))

    global_step = 0
    for epoch in range(args.epochs):
        model.train()
        for i, (a, p, r) in enumerate(train_loader):
            if args.max_train_batches and i >= args.max_train_batches: break
            loss = model(a, p, r) / args.grad_accum
            print(loss)
            loss.backward()
            if (i+1) % args.grad_accum == 0:
                nn.utils.clip_grad_norm_(model.adapter.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad(); global_step += 1
                if global_step % 50 == 0:
                    print(f"ep{epoch+1} step{global_step} loss {loss.item():.4f}")

        # ----- валидация WER --------------------------------------------------
        # model.eval(); refs, hyps = [], []
        # for (a, _, r) in val_loader:
        #     pred = model.generate_text(a[0])
        #     refs.append(r[0].lower())
        #     hyps.append(pred.lower())

        model.eval(); refs, hyps = [], []
        for j, (a, p, r) in enumerate(val_loader):
            if args.max_val_batches and j >= args.max_val_batches: break
            pred = model.generate_text(a[0], p[0]).lower()
            refs.append(r[0].lower()); hyps.append(pred)
    
            # печатаем первые 5 пар
            if j < 3:
                print("\n― PROMPT :", p[0][:80])
                print("  REF    :", r[0][:120])
                print("  HYP    :", pred[:120])
            
        wer_val = wer(refs, hyps)
        print(f"Epoch {epoch+1} | WER = {wer_val:.3f}")

        # save tiny-adapter
        os.makedirs(args.output_dir, exist_ok=True)
        torch.save(model.adapter.state_dict(),
                   f"{args.output_dir}/adapter_ep{epoch+1}.pt")

        gc.collect(); torch.cuda.empty_cache()

In [ ]:
main()

Загрузка датасета FLEURS ASR для ru_ru, сплит train...
Загрузка создание промптов для Fleurs ASR ...
Добавлен датасет FLEURS ASR с 2562 примерами
Загрузка датасета Common Voice для ru, сплит train...


Using the latest cached version of the module from C:\Users\ntads\.cache\huggingface\modules\datasets_modules\datasets\mozilla-foundation--common_voice_17_0\9d10386a731ff6e6ed4ec973a4dc204a9820e8c842fbe388bdba0dd205ed5016 (last modified on Sat Jun  7 19:51:14 2025) since it couldn't be found locally at mozilla-foundation/common_voice_17_0, or remotely on the Hugging Face Hub.


Загрузка создание промптов для Common Voice ...
Добавлен датасет Common Voice с 26377 примерами
Загрузка датасета CoVoST2 для перевода с ru на en, сплит train...
Загрузка создание промптов для CoVoST2 ...
Добавлен датасет CoVoST2 с 12112 примерами
Train batches: 4875 | Val examples: 2052
tensor(2.8293, device='cuda:0', grad_fn=<DivBackward0>)
tensor(2.7710, device='cuda:0', grad_fn=<DivBackward0>)
tensor(2.7139, device='cuda:0', grad_fn=<DivBackward0>)
tensor(2.8133, device='cuda:0', grad_fn=<DivBackward0>)
tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)
tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)
tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)
tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)
tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)
tensor(nan, device='cuda:0', grad_fn=<DivBackward0>)

― PROMPT : Переведи эту речь с русский на английский
  REF    : Outer space is indeed a new frontier for both developing and developed countries.
  HYP    : !!!!!!!!!!!

# DEBUG

In [ ]:
ds = UnifiedSpeechDataset(
    lang="ru",          # или "en"
    split="train",
    local_files_only=False,
    batch_size=1        # размер, который будет использоваться итератором __iter__
)

In [4]:
print("Всего примеров:", len(ds))
print("Ключи примера:", ds[0].keys())

Всего примеров: 41051
Ключи примера: dict_keys(['text_prompt', 'speech_input', 'text_response'])


In [12]:
ds[0]['text_response']

'К 17 сентября 1939 года польская оборона уже была прорвана, и единственной надеждой было отступление и реорганизация вдоль румынского плацдарма.'

In [13]:
ex = ds[0]
print("\n— text_prompt  :", ex["text_prompt"])
print("— speech_input :", ex["speech_input"][:10], "... (длина:", len(ex["speech_input"]), "sample-points)")
print("— text_response:", ex["text_response"])


— text_prompt  : Распознай эту речь на Russian
— speech_input : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] ... (длина: 445440 sample-points)
— text_response: К 17 сентября 1939 года польская оборона уже была прорвана, и единственной надеждой было отступление и реорганизация вдоль румынского плацдарма.


In [2]:
class Collator:
    def __call__(self, batch):
        audios    = [x["speech_input"]  for x in batch]
        prompts   = [x["text_prompt"]   for x in batch]
        responses = [x["text_response"] for x in batch]
        return audios, prompts, responses

# loader = DataLoader(ds, batch_size=2, collate_fn=Collator())

In [17]:
audios, prompts, responses = next(iter(loader))
print("\n=== Мини-батч ===")
print("audios   :", [len(a) for a in audios])   # формы waveforms
print("prompts  :", prompts)
print("responses:", responses)


=== Мини-батч ===
audios   : [445440, 317760]
prompts  : ['Распознай эту речь на Russian', 'Распознай эту речь на Russian']
responses: ['К 17 сентября 1939 года польская оборона уже была прорвана, и единственной надеждой было отступление и реорганизация вдоль румынского плацдарма.', 'Многие билеты, которые продаются в Интернете через такие аукционные сайты, как eBay или Craigslist, являются частично использованными многодневными билетами, которые позволяют посещать несколько парков в один день.']


In [3]:
class FCAdapterTiny(FCAdapter):
    """уменьшаем число параметров ≈ 120 k вместо 20 M"""
    def __init__(self):
        super().__init__(
            m=8,                 # stride 8 → сильное ужатие по времени
            d_x=1280,
            fc_layer_dim=256,    # было 11264
            llm_hidden_size=896,
        )

In [10]:
class DebugModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.proc = AutoProcessor.from_pretrained("openai/whisper-large-v3")
        self.whisper = AutoModelForSpeechSeq2Seq.from_pretrained(
            "openai/whisper-large-v3").model.encoder
        for p in self.whisper.parameters():
            p.requires_grad_(False)

        self.adapter = FCAdapterTiny()

        self.tok = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B",
                                                 use_fast=False)
        self.qwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B")
        for p in self.qwen.parameters():
            p.requires_grad_(False)
        self.text_emb = self.qwen.get_input_embeddings()

    # ----- «расширенный» forward с печатью -----------------------------------
    def forward(self, audios, prompts, responses):
        dev = next(self.adapter.parameters()).device

        # 1. Аудио → Whisper
        feats = self.proc.feature_extractor(
            audios,
            sampling_rate=16_000,
            return_tensors="pt",
            padding="max_length",              # всегда до фикс-длины
            max_length=480_000,                   # ровно то, что ждёт Whisper
            truncation=True                    # обрежет редкие >30-сек файлы
        ).input_features.to(dev)

        with torch.no_grad():
            h_w = self.whisper(feats).last_hidden_state
        print("Whisper out       :", h_w.shape)

        # 2. Adapter
        h_a = self.adapter(h_w)
        print("Adapter out       :", h_a.shape)

        # 3. Токенизация
        ids_p = self.tok(prompts, return_tensors="pt",
                         padding=True, add_special_tokens=False).input_ids.to(dev)
        ids_r = self.tok(responses, return_tensors="pt",
                         padding=True, add_special_tokens=False).input_ids.to(dev)
        with torch.no_grad():
            e_p = self.text_emb(ids_p)
            e_r = self.text_emb(ids_r)

        # 4. Конкатенация
        inp = torch.cat([h_a, e_p, e_r[:, :-1]], 1)
        print("Input concat      :", inp.shape)

        # ----- 4a. Attention mask --------------------------------------------
        att_audio   = torch.ones(h_a.shape[:2], device=dev, dtype=torch.long)
        att_prompt  = (ids_p != self.tok.pad_token_id).long()
        att_resp_in = (ids_r[:, :-1] != self.tok.pad_token_id).long()
        att_mask = torch.cat([att_audio, att_prompt, att_resp_in], 1)

        # Проверка длины
        assert att_mask.shape[1] == inp.shape[1]

        # ----- 4b. Labels -----------------------------------------------------
        labels_prefix = torch.full(
            (ids_p.size(0), h_a.size(1) + ids_p.size(1)),
            -100, dtype=torch.long, device=dev)
        # labels = torch.cat([labels_prefix, ids_r], 1)
        labels = torch.cat([labels_prefix, ids_r[:, 1:]], 1)   # <── FIX
        assert labels.shape[1] == inp.shape[1]

        # --- Debug prints ----------------------------------------------------
        print("\nattention_mask[0] :", att_mask[0][:30].tolist(), "…")
        print("labels[0]         :", labels[0][:30].tolist(), "…")
        print("(вырезка: -100 должны быть на аудио+промпте)")

        loss = self.qwen(inputs_embeds=inp,
                         attention_mask=att_mask,
                         labels=labels).loss
        return loss


# ---------- 2. Один батч + градиенты ----------------------------------------
def main():
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    # берём первые два примера, которыми вы делились
    ds = UnifiedSpeechDataset(lang="ru", split="train",
                              local_files_only=False, batch_size=1)
    subset = torch.utils.data.Subset(ds, [0, 1])

    class Collator:
        def __call__(self, batch):
            return ([x["speech_input"] for x in batch],
                    [x["text_prompt"] for x in batch],
                    [x["text_response"] for x in batch])

    loader = DataLoader(subset, batch_size=2, collate_fn=Collator())
    audios, prompts, responses = next(iter(loader))

    mdl = DebugModel().to(dev)
    print("\nПараметров адаптера:",
          sum(p.numel() for p in mdl.adapter.parameters()) / 1e3, "тыс.")

    loss = mdl(audios, prompts, responses)
    print("\nloss =", loss.item())

    loss.backward()                     # делаем шаг назад
    # Градиенты
    grad_adapter = any(p.grad is not None and p.grad.abs().sum() > 0
                       for p in mdl.adapter.parameters())
    grad_whisper = any(p.grad is not None and p.grad.abs().sum() > 0
                       for p in mdl.whisper.parameters())
    grad_qwen    = any(p.grad is not None and p.grad.abs().sum() > 0
                       for p in mdl.qwen.parameters())

    print("\nGradients — adapter:", grad_adapter,
          "| whisper:", grad_whisper,
          "| qwen:", grad_qwen)

    gc.collect(); torch.cuda.empty_cache()

In [11]:
main()

Загрузка датасета FLEURS ASR для ru_ru, сплит train...
Загрузка создание промптов для Fleurs ASR ...
Добавлен датасет FLEURS ASR с 2562 примерами
Загрузка датасета Common Voice для ru, сплит train...


Using the latest cached version of the module from C:\Users\ntads\.cache\huggingface\modules\datasets_modules\datasets\mozilla-foundation--common_voice_17_0\9d10386a731ff6e6ed4ec973a4dc204a9820e8c842fbe388bdba0dd205ed5016 (last modified on Sat Jun  7 19:51:14 2025) since it couldn't be found locally at mozilla-foundation/common_voice_17_0, or remotely on the Hugging Face Hub.


Загрузка создание промптов для Common Voice ...
Добавлен датасет Common Voice с 26377 примерами
Загрузка датасета CoVoST2 для перевода с ru на en, сплит train...
Загрузка создание промптов для CoVoST2 ...
Добавлен датасет CoVoST2 с 12112 примерами

Параметров адаптера: 569.728 тыс.
Whisper out       : torch.Size([2, 1500, 1280])
Adapter out       : torch.Size([2, 187, 896])
Input concat      : torch.Size([2, 252, 896])

attention_mask[0] : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] …
labels[0]         : [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100] …
(вырезка: -100 должны быть на аудио+промпте)

loss = 10.239734649658203

Gradients — adapter: True | whisper: False | qwen: False


In [ ]:
# train_speech2text_adapter.py
# -----------------------------------------------------------
"""
Обучение адаптера между Whisper-large-v3 и Qwen2-0.5B.
Сами большие модели остаются замороженными.

Запуск:
$ torchrun --nproc_per_node 4 train_speech2text_adapter.py \
    --adapter_type fc            # или transformer
"""

import os, math, argparse, gc, itertools, random
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoProcessor,               # для Whisper
    AutoModelForSpeechSeq2Seq,
    AutoTokenizer,               # для Qwen
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

from adapters import FCAdapter, TransformerAdapter
from unified_dataset import UnifiedSpeechDataset      # ваш код

# ---------- 1.  Параметры CLI -------------------------------------------------
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--batch_size", type=int, default=8)
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--adapter_type", choices=["fc", "transformer"],
                   default="fc")
    p.add_argument("--dataset_lang", choices=["ru", "en"], default="ru")
    p.add_argument("--output_dir", type=str, default="./checkpoints")
    p.add_argument("--grad_accum_steps", type=int, default=4)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--num_workers", type=int, default=4)
    return p.parse_args()

# ---------- 2.  Комбинированная модель ---------------------------------------
class Speech2TextModel(nn.Module):
    """
    Whisper (заморожен) -> ваш Adapter (учим) -> Qwen (заморожен).
    """

    def __init__(self, adapter_type: str = "fc"):
        super().__init__()

        # ----- Whisper --------------------------------------------------------
        self.processor = AutoProcessor.from_pretrained(
            "openai/whisper-large-v3", torch_dtype=torch.float16
        )
        self.whisper = AutoModelForSpeechSeq2Seq.from_pretrained(
            "openai/whisper-large-v3",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).model.encoder
        for p in self.whisper.parameters():
            p.requires_grad_(False)

        # ----- Adapter --------------------------------------------------------
        if adapter_type == "fc":
            self.adapter = FCAdapter()                # d_x=1280 → 896
        else:
            self.adapter = TransformerAdapter()       # d_x=1280 → 896

        # ----- Qwen -----------------------------------------------------------
        self.qwen_tok = AutoTokenizer.from_pretrained(
            "Qwen/Qwen2-0.5B", use_fast=False
        )
        self.qwen = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2-0.5B",
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
        )
        for p in self.qwen.parameters():
            p.requires_grad_(False)

        self.qwen_embed = self.qwen.get_input_embeddings()  # вес не обучаем

    # -------- forward --------------------------------------------------------
    def forward(self, audios, prompts: List[str], responses: List[str]):
        """
        audios      : List[np.ndarray] — PCM 16 kHz waveforms
        prompts     : текстовые промпты (строки)
        responses   : референс-транскрипт/перевод
        """

        device = next(self.adapter.parameters()).device
        # -- 2.1  Аудио → мел-спектры → Whisper-encoder ------------------------
        with torch.no_grad():
            feats = self.processor.feature_extractor(
                audios, sampling_rate=16_000,
                return_tensors="pt", padding=True
            ).input_features.to(device).to(torch.float16)
            w_out = self.whisper(feats).last_hidden_state  # (B, T_w, 1280)

        # -- 2.2  Adapter ------------------------------------------------------
        a_out = self.adapter(w_out)                       # (B, T_a, 896)

        # -- 2.3  Токены промпта и ответа -------------------------------------
        tok_prompt = self.qwen_tok(
            prompts, return_tensors="pt", padding=True,
            add_special_tokens=False
        ).input_ids.to(device)
        tok_resp = self.qwen_tok(
            responses, return_tensors="pt", padding=True,
            add_special_tokens=False
        ).input_ids.to(device)

        # Эмбеддинги для текста (градиенты не нужны)
        with torch.no_grad():
            p_emb = self.qwen_embed(tok_prompt)           # (B, T_p, 896)
            r_emb = self.qwen_embed(tok_resp)             # (B, T_r, 896)

        # -- 2.4  Склейка ------------------------------------------------------
        inp_emb = torch.cat([a_out, p_emb, r_emb[:, :-1]], dim=1)
        seq_len = inp_emb.size(1)

        # attention_mask: 1 на содержимых позициях, 0 — паддинг
        att_audio   = torch.ones(a_out.shape[:2],  dtype=torch.long, device=device)
        att_prompt  = (tok_prompt != self.qwen_tok.pad_token_id).long()
        att_resp_in = (tok_resp[:, :-1] != self.qwen_tok.pad_token_id).long()
        att_mask = torch.cat([att_audio, att_prompt, att_resp_in], dim=1)

        # -- 2.5  Labels (mask для префикса) ----------------------------------
        labels_prefix = torch.full(
            (tok_prompt.size(0), a_out.size(1) + tok_prompt.size(1)),
            -100, dtype=torch.long, device=device
        )
        labels = torch.cat([labels_prefix, tok_resp], dim=1)

        # -- 2.6  LLM ----------------------------------------------------------
        loss = self.qwen(
            inputs_embeds=inp_emb,
            attention_mask=att_mask,
            labels=labels
        ).loss
        return loss

# ---------- 3.  Data collator -------------------------------------------------
class Collator:
    def __call__(self, batch):
        audios       = [item["speech_input"]  for item in batch]
        prompts      = [item["text_prompt"]   for item in batch]
        responses    = [item["text_response"] for item in batch]
        return audios, prompts, responses

# ---------- 4.  Тренировочный цикл -------------------------------------------
def train():
    args = parse_args()
    torch.manual_seed(args.seed)

    ds = UnifiedSpeechDataset(
        lang=args.dataset_lang, split="train", local_files_only=False,
        batch_size=args.batch_size
    )
    train_loader = DataLoader(
        ds, batch_size=args.batch_size, shuffle=True,
        collate_fn=Collator(), num_workers=args.num_workers, pin_memory=True
    )

    model = Speech2TextModel(adapter_type=args.adapter_type)
    model = model.to(torch.bfloat16).cuda()

    # Обучаем ТОЛЬКО adapter.* параметры
    opt = AdamW(model.adapter.parameters(), lr=args.lr, weight_decay=1e-2)
    steps_per_epoch = math.ceil(len(ds) / args.batch_size / args.grad_accum_steps)
    scheduler = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=steps_per_epoch//10,
        num_training_steps=steps_per_epoch*args.epochs
    )

    model.train()
    global_step = 0
    for epoch in range(args.epochs):
        for i, (audios, prompts, responses) in enumerate(train_loader):
            loss = model(audios, prompts, responses) / args.grad_accum_steps
            loss.backward()

            if (i + 1) % args.grad_accum_steps == 0:
                nn.utils.clip_grad_norm_(model.adapter.parameters(), 1.0)
                opt.step()
                scheduler.step()
                opt.zero_grad()
                global_step += 1

            if global_step % 50 == 0:
                print(f"epoch {epoch+1} | step {global_step} | "
                      f"loss {loss.item():.4f}")

        # чек-пойнт адаптера
        os.makedirs(args.output_dir, exist_ok=True)
        torch.save(model.adapter.state_dict(),
                   f"{args.output_dir}/adapter_step{global_step}.pt")

        gc.collect(); torch.cuda.empty_cache()

if __name__ == "__main__":
    train()
